In [ ]:
import os

# ─── PATHS ────────────────────────────────────────────────────────────────────
BASE_DIR        = os.path.dirname(os.path.abspath("model"))
DATA_DIR        = os.path.join(BASE_DIR, "data")
CHECKPOINTS_DIR = os.path.join(BASE_DIR, "checkpoints")
LOGS_DIR        = os.path.join(BASE_DIR, "logs")
RESULTS_DIR     = os.path.join(BASE_DIR, "results")

for d in [DATA_DIR, CHECKPOINTS_DIR, LOGS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ─── KAGGLE DATASET SLUGS ─────────────────────────────────────────────────────
KAGGLE_DATASETS = {
    "weapons":   "issaisasank/guns-object-detection",   # YOLO-format weapon images
    "violence":  "mohamedmustafa/real-life-violence-situations-dataset",
    "emotion":   "msambare/fer2013",                    # FER2013 facial expression
    "anomaly":   "odins0n/ucf-crime-dataset-small",     # UCF-Crime subset
}
# NOTE: set your Kaggle API credentials before running Part 1:
#   export KAGGLE_USERNAME=your_username
#   export KAGGLE_KEY=your_api_key
# OR place kaggle.json in ~/.kaggle/

# ─── MODEL CHECKPOINTS (output names) ─────────────────────────────────────────
WEAPON_BEST_MODEL    = os.path.join(CHECKPOINTS_DIR, "weapon_best.pt")
VIOLENCE_BEST_MODEL  = os.path.join(CHECKPOINTS_DIR, "violence_best.pt")
EMOTION_BEST_MODEL   = os.path.join(CHECKPOINTS_DIR, "emotion_best.pt")
FUSION_BEST_MODEL    = os.path.join(CHECKPOINTS_DIR, "fusion_best.pt")

# ─── TRAINING HYPERPARAMETERS ─────────────────────────────────────────────────
SEED = 42

WEAPON_CFG = {
    "model":        "yolov8n.pt",   # nano — fastest, pretrained on COCO
    "epochs":       80,
    "imgsz":        416,
    "batch":        16,
    "lr0":          0.001,
    "patience":     15,             # early stopping
    "conf_thresh":  0.5,
    "iou_thresh":   0.45,
}

VIOLENCE_CFG = {
    "backbone":     "mobilenet_v2", # pretrained on ImageNet
    "seq_len":      32,             # frames per clip
    "frame_size":   (112, 112),
    "lstm_hidden":  256,
    "lstm_layers":  2,
    "dropout":      0.4,
    "epochs":       60,
    "batch":        8,
    "lr":           3e-4,
    "weight_decay": 1e-4,
    "patience":     12,
    "num_classes":  2,              # violent / non-violent
}

EMOTION_CFG = {
    "backbone":     "efficientnet_b0",  # pretrained on ImageNet
    "img_size":     (48, 48),
    "epochs":       50,
    "batch":        64,
    "lr":           1e-3,
    "weight_decay": 1e-4,
    "patience":     10,
    "num_classes":  7,              # FER2013: angry,disgust,fear,happy,sad,surprise,neutral
    "fear_class":   2,              # index of 'fear' in FER2013
}

FUSION_CFG = {
    "epochs":           40,
    "batch":            32,
    "lr":               1e-3,
    "weight_decay":     1e-4,
    "patience":         10,
    # Predetermined threat weights (priority order: violence>weapons>precursor>emotion)
    # These are used as INITIALISATION if learn_weights=True,
    # or as fixed values if learn_weights=False.
    "threat_weights": {
        "violence":  0.40,
        "weapon":    0.30,
        "precursor": 0.20,
        "emotion":   0.10,
    },
    "learn_weights": False,   # Set True to let model learn weights; False = fixed
}

# ─── THREAT SCORING THRESHOLDS ────────────────────────────────────────────────
THREAT = {
    "low":    0.4,   # below → log only
    "medium": 0.7,   # above → alert dashboard
    # above medium → immediate alert
    "loitering_secs":   180,
    "concealment_secs":   5,
    "gaze_avoid_secs":   10,
    "proximity_meters":   2,
}

# ─── DEVICE ───────────────────────────────────────────────────────────────────
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Config] Using device: {DEVICE}")


In [ ]:
!pip install ultralytics torch torchvision tqdm scikit-learn opencv-python pandas matplotlib

In [ ]:

import os, shutil, zipfile, random, csv
from pathlib import Path
import cv2
import numpy as np
from tqdm import tqdm

# ── import project config ──────────────────────────────────────────────────────
import sys; sys.path.insert(0, os.path.dirname("/kaggle/working/"))
#from config import (DATA_DIR, KAGGLE_DATASETS,
 #                   VIOLENCE_CFG, EMOTION_CFG)

# ── 1. Kaggle download helper ──────────────────────────────────────────────────
def download_kaggle(slug: str, dest: str):
    """Download and unzip a Kaggle dataset."""
    import kagglehub          # pip install kagglehub
    path = kagglehub.dataset_download(slug)
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    src = Path(path)
    # kagglehub downloads to a cache; copy to our data dir
    if src != dest:
        shutil.copytree(src, dest, dirs_exist_ok=True)
    print(f"[Data] '{slug}' ready at {dest}")
    return str(dest)


# ── 2. Weapon dataset (already YOLO-formatted) ────────────────────────────────
def prepare_weapons():
    dest = os.path.join(DATA_DIR, "weapons")
    if Path(dest).exists() and any(Path(dest).iterdir()):
        print("[Data] Weapons dataset already prepared. Skipping.")
        return dest
    download_kaggle(KAGGLE_DATASETS["weapons"], dest)
    # Expected structure after download:
    #   weapons/
    #     images/train/  images/val/
    #     labels/train/  labels/val/
    #     data.yaml
    # Generate data.yaml if missing
    yaml_path = os.path.join(dest, "data.yaml")
    if not os.path.exists(yaml_path):
        _write_weapon_yaml(dest, yaml_path)
    print(f"[Data] Weapons ready → {dest}")
    return dest


def _write_weapon_yaml(data_dir, yaml_path):
    """Create YOLO data.yaml for the weapons dataset."""
    import yaml
    cfg = {
        "path":  data_dir,
        "train": "images/train",
        "val":   "images/val",
        "nc":    5,
        "names": ["handgun", "rifle", "knife", "grenade", "other_weapon"],
    }
    with open(yaml_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print(f"[Data] Wrote {yaml_path}")


# ── 3. Violence dataset ────────────────────────────────────────────────────────
def prepare_violence():
    """
    Real-Life Violence Situations dataset:
    Violence/    → labelled violence
    NonViolence/ → labelled non-violence
    Converts video folders into frame sequences stored as:
      data/violence/frames/{train,val,test}/{violence,nonviolence}/{clip_id}/{frame_N}.jpg
    """
    dest_frames = os.path.join(DATA_DIR, "violence", "frames")
    if Path(dest_frames).exists():
        print("[Data] Violence frames already prepared. Skipping.")
        return dest_frames
    
    raw_dir = download_kaggle(KAGGLE_DATASETS["violence"],
                              os.path.join(DATA_DIR, "violence", "raw"))
    raw_dir = "/kaggle/working/data/violence/raw/real life violence situations/Real Life Violence Dataset"
    _extract_violence_frames(raw_dir, dest_frames)
    return dest_frames


def _extract_violence_frames(raw_dir: str, frames_dir: str,
                              seq_len: int = 32,
                              val_split: float = 0.15,
                              test_split: float = 0.10):
    """Extract fixed-length frame sequences from each video."""
    classes = {"Violence": 1, "NonViolence": 0}
    splits  = {"train": [], "val": [], "test": []}

    for class_name, label in classes.items():
        class_path = Path(raw_dir) / class_name
        if not class_path.exists():
            # Try lowercase or flat structure
            class_path = Path(raw_dir) / class_name.lower()
        videos = sorted(list(class_path.glob("*.mp4")) +
                        list(class_path.glob("*.avi")) +
                        list(class_path.glob("*.mkv")))
        random.seed(42)
        random.shuffle(videos)
        n  = len(videos)
        nv = int(n * val_split)
        nt = int(n * test_split)
        split_map = (
            [("train", v) for v in videos[:n - nv - nt]] +
            [("val",   v) for v in videos[n - nv - nt:n - nt]] +
            [("test",  v) for v in videos[n - nt:]]
        )
        for split, vid_path in tqdm(split_map,
                                    desc=f"Extracting {class_name}"):
            clip_id  = vid_path.stem
            out_dir  = Path(frames_dir) / split / class_name.lower() / clip_id
            out_dir.mkdir(parents=True, exist_ok=True)
            frames   = _sample_frames(str(vid_path), seq_len,
                                      VIOLENCE_CFG["frame_size"])
            for i, frame in enumerate(frames):
                cv2.imwrite(str(out_dir / f"frame_{i:04d}.jpg"), frame)

    print(f"[Data] Violence frames saved → {frames_dir}")


def _sample_frames(video_path: str, n_frames: int, size: tuple):
    cap    = cv2.VideoCapture(video_path)
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return [np.zeros((*size, 3), dtype=np.uint8)] * n_frames
    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    frames  = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            frame = np.zeros((*size, 3), dtype=np.uint8)
        else:
            frame = cv2.resize(frame, size)
        frames.append(frame)
    cap.release()
    return frames


# ── 4. Emotion / FER2013 ──────────────────────────────────────────────────────
def prepare_emotion():
    """
    Kaggle-preprocessed FER2013:
    Already exists as data/emotion/{train,test}/{class_name}/*.jpg
    """
    # 1. Define where the data actually lives on Kaggle
    # Update this path if your Kaggle input folder name is different
    kaggle_input_path = "/kaggle/input/datasets/msambare/fer2013" 
    
    dest = os.path.join(DATA_DIR, "emotion")
    
    # 2. If the data is already in our working directory, skip
    if Path(dest).exists() and len(list(Path(dest).rglob("*.jpg"))) > 1000:
        print("[Data] Emotion dataset already prepared in working directory. Skipping.")
        return dest

    # 3. Instead of looking for a CSV, we copy the existing folders
    print(f"[Data] Copying pre-structured FER2013 images from {kaggle_input_path}...")
    
    if os.path.exists(kaggle_input_path):
        shutil.copytree(kaggle_input_path, dest, dirs_exist_ok=True)
        print(f"[Data] Emotion images ready at → {dest}")
    else:
        print(f"⚠️ Error: Could not find emotion data at {kaggle_input_path}. Check the folder name in Kaggle Input.")
        
    return dest
    


# ── 5. Build multi-label fusion manifest ──────────────────────────────────────
def build_fusion_manifest(violence_frames_dir: str, out_csv: str):
    """
    Creates a CSV manifest for fusion training.
    Columns: clip_path, label_violence, label_weapon (placeholder), label_emotion (placeholder)
    Only violence labels are real here; weapon/emotion are set to -1 (unlabelled)
    and filled in at inference time by the respective module heads.
    """
    rows = []
    for split in ["train", "val", "test"]:
        for cls in ["violence", "nonviolence"]:
            clip_dir = Path(violence_frames_dir) / split / cls
            if not clip_dir.exists():
                continue
            for clip_path in sorted(clip_dir.iterdir()):
                rows.append({
                    "clip_path":       str(clip_path),
                    "split":           split,
                    "label_violence":  1 if cls == "violence" else 0,
                    "label_weapon":    -1,
                    "label_emotion":   -1,
                })
    import pandas as pd
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"[Data] Fusion manifest → {out_csv}  ({len(rows)} clips)")


# ── MAIN ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 60)
    print("ProactiveGuard — Part 1: Dataset Preparation")
    print("=" * 60)

    weapon_dir  = prepare_weapons()
    violence_dir = prepare_violence()
    emotion_dir  = prepare_emotion()

    manifest = os.path.join(DATA_DIR, "fusion_manifest.csv")
    build_fusion_manifest(violence_dir, manifest)

    print("\n✅  All datasets ready.")
    print(f"   Weapons  → {weapon_dir}")
    print(f"   Violence → {violence_dir}")
    print(f"   Emotion  → {emotion_dir}")
    print(f"   Manifest → {manifest}")


In [ ]:
"""
Part 2b — Violence Detection Training  [Run on DEVICE B]
MobileNetV2 (pretrained) + Bi-LSTM + Self-Attention for temporal violence detection.

Dependencies:
    pip install torch torchvision tqdm scikit-learn
Output:
    checkpoints/violence_best.pt
"""
import os, sys, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, f1_score, precision_recall_curve,
                              average_precision_score)

sys.path.insert(0, os.path.dirname("/kaggle/working/"))
#from config import (DATA_DIR, CHECKPOINTS_DIR, LOGS_DIR,
 #                   VIOLENCE_CFG, VIOLENCE_BEST_MODEL, DEVICE, SEED)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)


# ═══════════════════════════════════════════════════════════════════════════════
#  DATASET
# ═══════════════════════════════════════════════════════════════════════════════
class ViolenceFrameDataset(Dataset):
    """
    Loads pre-extracted frame sequences from:
      data/violence/frames/{split}/{violence|nonviolence}/{clip_id}/*.jpg
    Returns a tensor of shape (seq_len, C, H, W) and an integer label.
    """
    CLASSES = {"nonviolence": 0, "violence": 1}

    def __init__(self, frames_root: str, split: str = "train",
                 seq_len: int = 32, augment: bool = False):
        self.seq_len  = seq_len
        self.augment  = augment
        self.clips    = []   # list of (clip_dir_path, label)

        base = Path(frames_root) / split
        for cls_name, label in self.CLASSES.items():
            cls_dir = base / cls_name
            if not cls_dir.exists():
                continue
            for clip_dir in sorted(cls_dir.iterdir()):
                frames = sorted(clip_dir.glob("*.jpg"))
                if len(frames) >= 4:
                    self.clips.append((clip_dir, label))

        # ── transforms ────────────────────────────────────────────────────────
        self.base_tfm = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(VIOLENCE_CFG["frame_size"]),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std =[0.229, 0.224, 0.225]),
        ])
        self.aug_tfm = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(VIOLENCE_CFG["frame_size"]),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                   saturation=0.2, hue=0.05),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std =[0.229, 0.224, 0.225]),
        ])

    def __len__(self): return len(self.clips)

    def __getitem__(self, idx):
        import cv2
        clip_dir, label = self.clips[idx]
        frames = sorted(clip_dir.glob("*.jpg"))
        # Sample seq_len frames uniformly
        indices = np.linspace(0, len(frames) - 1, self.seq_len, dtype=int)
        tfm = self.aug_tfm if self.augment else self.base_tfm
        tensors = []
        for i in indices:
            img = cv2.imread(str(frames[i]))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            tensors.append(tfm(img))
        clip_tensor = torch.stack(tensors)  # (seq_len, C, H, W)
        return clip_tensor, label


# ═══════════════════════════════════════════════════════════════════════════════
#  MODEL: CNN-Feature-Extractor + Bi-LSTM + Self-Attention
# ═══════════════════════════════════════════════════════════════════════════════
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1)  # *2 for bidirectional

    def forward(self, lstm_out):
        # lstm_out: (batch, seq, hidden*2)
        scores  = self.attn(lstm_out).squeeze(-1)       # (batch, seq)
        weights = F.softmax(scores, dim=-1)              # (batch, seq)
        context = (weights.unsqueeze(-1) * lstm_out).sum(dim=1)  # (batch, hidden*2)
        return context, weights


class ViolenceDetector(nn.Module):
    """
    Architecture:
      MobileNetV2 (pretrained, frozen first 10 layers) → feature per frame
      → Bi-LSTM(256, 2 layers)
      → Temporal Self-Attention
      → Classifier head
    """
    def __init__(self, num_classes: int = 2,
                 lstm_hidden: int = 256,
                 lstm_layers: int = 2,
                 dropout: float = 0.4):
        super().__init__()
        self.lstm_hidden = lstm_hidden

        # ── CNN backbone (MobileNetV2 pretrained on ImageNet) ──────────────
        backbone = models.mobilenet_v2(
            weights=models.MobileNet_V2_Weights.IMAGENET1K_V1
        )
        # Remove classifier; keep feature extraction
        self.cnn = backbone.features  # output: (batch, 1280, H//32, W//32)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))  # → (batch, 1280)
        self.feat_dim = 1280

        # Freeze early layers to preserve pretrained spatial features
        for i, layer in enumerate(self.cnn):
            if i < 10:
                for p in layer.parameters():
                    p.requires_grad = False

        # ── Temporal encoder ────────────────────────────────────────────────
        self.lstm = nn.LSTM(
            input_size  = self.feat_dim,
            hidden_size = lstm_hidden,
            num_layers  = lstm_layers,
            batch_first = True,
            bidirectional = True,
            dropout     = dropout if lstm_layers > 1 else 0.0,
        )
        self.attention = TemporalAttention(lstm_hidden)

        # ── Classifier ──────────────────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # x: (batch, seq_len, C, H, W)
        B, T, C, H, W = x.shape
        # Flatten batch and time for CNN
        x_flat  = x.view(B * T, C, H, W)
        feats   = self.pool(self.cnn(x_flat)).squeeze(-1).squeeze(-1)  # (B*T, 1280)
        feats   = feats.view(B, T, self.feat_dim)                       # (B, T, 1280)

        lstm_out, _ = self.lstm(feats)                                  # (B, T, H*2)
        context, attn_weights = self.attention(lstm_out)                # (B, H*2)
        logits  = self.classifier(context)                              # (B, num_classes)
        return logits, attn_weights

    def extract_features(self, x):
        """Return feature embedding (for fusion) and attention weights."""
        logits, attn = self.forward(x)
        probs = F.softmax(logits, dim=-1)
        return probs[:, 1], attn   # violence probability scalar per sample


# ═══════════════════════════════════════════════════════════════════════════════
#  TRAINING LOOP
# ═══════════════════════════════════════════════════════════════════════════════
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for clips, labels in tqdm(loader, desc="  Train", leave=False):
        clips, labels = clips.to(device), labels.to(device)
        optimizer.zero_grad()
        logits, _ = model(clips)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * clips.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total   += clips.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []
    for clips, labels in tqdm(loader, desc="  Val  ", leave=False):
        clips, labels = clips.to(device), labels.to(device)
        logits, _ = model(clips)
        loss = criterion(logits, labels)
        probs = F.softmax(logits, dim=-1)[:, 1]
        total_loss += loss.item() * clips.size(0)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_probs.extend(probs.cpu().tolist())
    n   = len(all_labels)
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / n
    f1  = f1_score(all_labels, all_preds, average="binary", zero_division=0)
    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / n, acc, f1, auc, all_preds, all_labels, all_probs


def train_violence_detector():
    cfg = VIOLENCE_CFG
    frames_dir = os.path.join(DATA_DIR, "violence", "frames")
    if not Path(frames_dir).exists():
        raise RuntimeError("Run 01_data_preparation.py first.")

    print("=" * 60)
    print("Part 2b — Violence Detection (MobileNetV2 + Bi-LSTM)")
    print(f"Device: {DEVICE}")
    print("=" * 60)

    # ── Datasets ───────────────────────────────────────────────────────────────
    train_ds = ViolenceFrameDataset(frames_dir, "train", cfg["seq_len"], augment=True)
    val_ds   = ViolenceFrameDataset(frames_dir, "val",   cfg["seq_len"], augment=False)
    test_ds  = ViolenceFrameDataset(frames_dir, "test",  cfg["seq_len"], augment=False)
    print(f"  Train clips: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")

    # Class-weighted sampler to handle imbalance
    labels  = [s[1] for s in train_ds.clips]
    counts  = np.bincount(labels)
    weights = 1.0 / counts[labels]
    sampler = torch.utils.data.WeightedRandomSampler(weights, len(weights))

    train_loader = DataLoader(train_ds, batch_size=cfg["batch"],
                              sampler=sampler, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg["batch"],
                              shuffle=False, num_workers=4)
    test_loader  = DataLoader(test_ds,  batch_size=cfg["batch"],
                              shuffle=False, num_workers=4)

    # ── Model / Optimiser ──────────────────────────────────────────────────────
    device = torch.device(DEVICE)
    model  = ViolenceDetector(
        num_classes  = cfg["num_classes"],
        lstm_hidden  = cfg["lstm_hidden"],
        lstm_layers  = cfg["lstm_layers"],
        dropout      = cfg["dropout"],
    ).to(device)

    # Label-smoothed cross-entropy
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg["lr"], weight_decay=cfg["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg["epochs"], eta_min=1e-6
    )

    # ── Training loop with early stopping ─────────────────────────────────────
    best_f1     = 0.0
    no_improve  = 0
    log_path    = os.path.join(LOGS_DIR, "violence_training.csv")
    os.makedirs(LOGS_DIR, exist_ok=True)

    with open(log_path, "w") as log:
        log.write("epoch,train_loss,train_acc,val_loss,val_acc,val_f1,val_auc,lr\n")

        for epoch in range(1, cfg["epochs"] + 1):
            t0 = time.time()
            tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, device)
            vl_loss, vl_acc, vl_f1, vl_auc, _, _, _ = eval_epoch(
                model, val_loader, criterion, device
            )
            scheduler.step()
            elapsed = time.time() - t0
            lr_now  = optimizer.param_groups[0]["lr"]

            print(f"Epoch {epoch:3d}/{cfg['epochs']} | "
                  f"Train Loss {tr_loss:.4f} Acc {tr_acc:.4f} | "
                  f"Val Loss {vl_loss:.4f} Acc {vl_acc:.4f} "
                  f"F1 {vl_f1:.4f} AUC {vl_auc:.4f} | "
                  f"LR {lr_now:.2e} | {elapsed:.0f}s")
            log.write(f"{epoch},{tr_loss:.5f},{tr_acc:.5f},"
                      f"{vl_loss:.5f},{vl_acc:.5f},{vl_f1:.5f},{vl_auc:.5f},{lr_now:.2e}\n")
            log.flush()

            # Best model saving
            if vl_f1 > best_f1:
                best_f1    = vl_f1
                no_improve = 0
                torch.save({
                    "epoch":        epoch,
                    "model_state":  model.state_dict(),
                    "optim_state":  optimizer.state_dict(),
                    "val_f1":       vl_f1,
                    "val_auc":      vl_auc,
                    "val_acc":      vl_acc,
                    "config":       cfg,
                }, VIOLENCE_BEST_MODEL)
                print(f"  ✅  New best model saved (F1={vl_f1:.4f})")
            else:
                no_improve += 1
                if no_improve >= cfg["patience"]:
                    print(f"  Early stopping triggered at epoch {epoch}.")
                    break

    # ── Final test evaluation ──────────────────────────────────────────────────
    print("\n--- Final Test Evaluation ---")
    ckpt = torch.load(VIOLENCE_BEST_MODEL, map_location=device, weights_only = False)
    model.load_state_dict(ckpt["model_state"])
    _, ts_acc, ts_f1, ts_auc, preds, labels_, probs = eval_epoch(
        model, test_loader, criterion, device
    )
    print(f"  Test Accuracy: {ts_acc:.4f}")
    print(f"  Test F1      : {ts_f1:.4f}")
    print(f"  Test AUC-ROC : {ts_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(labels_, preds,
                                 target_names=["NonViolence", "Violence"],
                                 digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(labels_, preds))

    # Precision-Recall AUC
    ap = average_precision_score(labels_, probs)
    print(f"  Average Precision (PR-AUC): {ap:.4f}")

    print(f"\n📦 Transfer {VIOLENCE_BEST_MODEL} to the fusion device.")


# ── MAIN ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    train_violence_detector()
